# Exploratory Data Analysis of internal cfdna data
- samples provided by Comenius University Science Park

In [10]:
import pandas as pd
import os

In [11]:
base_path = "/data/projects/liquid_biopsy/Projects/cfDNA/cfDNA/"
path = "/data/projects/liquid_biopsy/Projects/cfDNA/cfDNA/data/source/internal/"
descriptions = pd.read_csv(path + "Onkologické vzorky - archív - Značenie.tsv", sep="\t")
dataset = pd.read_csv(path + "Onkologické vzorky - archív - Dataset.tsv", sep="\t")
cancer_types = pd.read_csv(path + "Onkologické vzorky - archív - Typ rakoviny.tsv", sep="\t")
material = pd.read_csv(path + "Onkologické vzorky - archív - Materiál.tsv", sep="\t")

In [12]:
descriptions['Dataset'].value_counts()

Dataset
Lynch populačná     501
Lynch CRC           227
Lynch rodiny         90
White Spring         23
MSI panel             4
Populačná EDTA        2
Populačná STRECK      1
Name: count, dtype: int64

In [13]:
ly_metadata = pd.read_csv(base_path + 'data/manifest/internal_metadata_ly.csv')
ly_metadata['cancer_true'].value_counts()


cancer_true
0    473
1    216
Name: count, dtype: int64

In [14]:
gs_metadata = pd.read_csv(base_path + 'data/manifest/internal_metadata_gs.csv')
gs_metadata['cancer_true'].value_counts()

cancer_true
0    19
1    19
Name: count, dtype: int64

In [15]:
gs_metadata

,sample_id,disease,dataset,material,frag_path,stage,cancer_true
0,gspp_ctr_00238_01_01_pl,ctr,gspp,pl,frag/gspp_ctr_00238_01_01_pl.GRCh37.frag.bed.gz,NaN,0
1,gsca_pca_00017_01_01_pl,pca,gsca,pl,frag/gsca_pca_00017_01_01_pl.GRCh37.frag.bed.gz,NaN,1
2,gspp_ctr_00019_01_01_pl,ctr,gspp,pl,frag/gspp_ctr_00019_01_01_pl.GRCh37.frag.bed.gz,NaN,0
3,gspp_ctr_00133_01_01_pl,ctr,gspp,pl,frag/gspp_ctr_00133_01_01_pl.GRCh37.frag.bed.gz,NaN,0
4,gsca_pca_00011_01_01_pl,pca,gsca,pl,frag/gsca_pca_00011_01_01_pl.GRCh37.frag.bed.gz,NaN,1
5,gsca_pca_00125_01_01_pl,pca,gsca,pl,frag/gsca_pca_00125_01_01_pl.GRCh37.frag.bed.gz,NaN,1
6,gspp_ctr_00300_01_01_pl,ctr,gspp,pl,frag/gspp_ctr_00300_01_01_pl.GRCh37.frag.bed.gz,NaN,0
7,gsca_pca_00009_01_01_pl,pca,gsca,pl,frag/gsca_pca_00009_01_01_pl.GRCh37.frag.bed.gz,NaN,1
8,gspp_ctr_00228_01_01_pl,ctr,gspp,pl,frag/gspp_ctr_00228_01_01_pl.GRCh37.frag.bed.gz,NaN,0
9,gsca_pca_00123_01_01_pl,pca,gsca,pl,frag/gsca_pca_00123_01_01_pl.GRCh37.frag.bed.gz,NaN,1


In [16]:
sample_ids = ly_metadata['sample_id'].unique()
dataset, typ, id, sample_order, replicate, material = zip(*[s.split('_') for s in sample_ids])
sample_order = [int(i) for i in sample_order]
replicate = [int(i) for i in replicate]
ly_metadata['sample_order'] = sample_order
ly_metadata['id'] = id
ly_metadata['replicate'] = replicate
samples_keep = ly_metadata.sort_values('sample_order').drop_duplicates('id', keep='last')['sample_id']
ly_metadata_filtered = ly_metadata[ly_metadata['sample_id'].isin(samples_keep)]
ly_metadata_filtered.shape
ly_metadata_filtered = ly_metadata_filtered.drop(columns=['sample_order', 'replicate', 'id'])
# ly_metadata_filtered.to_csv(base_path + 'data/manifest/internal_metadata_ly_filtered.csv', index=False)


In [17]:
import random
random.seed(1)
ly_samples = pd.read_csv(base_path + 'data/manifest/internal_metadata_ly_filtered.csv')
ly_samples = ly_samples['sample_id']
# use 10% for testing
test_samples = random.sample(list(ly_samples), int(0.2 * len(ly_samples)))
train_samples = [s for s in ly_samples if s not in test_samples]
test_metadata = ly_metadata_filtered[ly_metadata_filtered['sample_id'].isin(test_samples)]
train_metadata = ly_metadata_filtered[ly_metadata_filtered['sample_id'].isin(train_samples)]
test_metadata.to_csv(base_path + 'data/manifest/internal_metadata_ly_filtered_test.csv', index=False)
train_metadata.to_csv(base_path + 'data/manifest/internal_metadata_ly_filtered_train.csv', index=False)

In [24]:
display(train_metadata['cancer_true'].value_counts())
display(test_metadata['cancer_true'].value_counts())

cancer_true
0    381
1     84
Name: count, dtype: int64

cancer_true
0    92
1    24
Name: count, dtype: int64

In [19]:
# display(descriptions['Dataset'].value_counts())
plasma_samples = descriptions[descriptions['Materiál'] == "Plazma"].reset_index(drop=True)
print(plasma_samples['Dataset'].value_counts())
# print(f'Number of unique patients: {plasma_samples["Pacient"].nunique()}')
# print(plasma_samples['Poradie odberu'].value_counts())
# first_sampling = plasma_samples[plasma_samples['Poradie odberu'] == 2]
# print(f'Number of unique patients with first sampling: {first_sampling["Pacient"].nunique()}')
# print(first_sampling['Dataset'].value_counts())

Dataset
Lynch populačná    498
Lynch CRC          224
Lynch rodiny        90
Populačná EDTA       2
Name: count, dtype: int64


In [20]:
df_lynch_crc = plasma_samples[plasma_samples['Dataset'] == "Lynch CRC"].reset_index(drop=True)
df_population_lynch = plasma_samples[plasma_samples['Dataset'] == "Lynch populačná"].reset_index(drop=True)
# df_lynch_families = plasma_samples[plasma_samples['Dataset'] == "Lynch rodiny"].reset_index(drop=True)
# df_population_edta = plasma_samples[plasma_samples['Dataset'] == "Populačná EDTA"].reset_index(drop=True)
print(len(df_lynch_crc), len(df_population_lynch))

224 498


In [21]:
df_lynch_crc['Sekvenačný beh'] = pd.to_datetime(df_lynch_crc['Sekvenačný beh'])

In [22]:
df_lynch_crc.groupby(['Sekvenačný beh', 'Poradie odberu'])['Pacient'].unique()

Sekvenačný beh  Poradie odberu
2022-06-10      1                                      [1vwhw, k0uya, ecf5u, t4mtl]
                2                                      [1vwhw, k0uya, ecf5u, t4mtl]
2022-06-21      1                               [4tmpe, zdsrl, 4tqlh, dwv48, up7uu]
                2                               [4tmpe, zdsrl, 4tqlh, dwv48, up7uu]
2022-06-29      1                                             [sx7fp, 2wmor, rpqe0]
                2                                             [sx7fp, 2wmor, rpqe0]
2022-07-08      1                                      [cgyd0, emejb, ktnly, pjo8j]
                2                                      [cgyd0, emejb, ktnly, pjo8j]
2022-07-22      1                                             [5jmb0, fn44r, ab2ku]
                2                                             [5jmb0, fn44r, ab2ku]
2022-07-27      1                                      [f3wyc, zgtwr, zcncr, bzxra]
                2                            

In [23]:
# pozriet sa na gsca gspp tento groupby (datumy+ repliky)

# TODO feature motivy, snakemake - gsca gspp